# Alert escalation prediction: reproducible pipeline
Predicts the probability that an AML monitoring alert (`signal_id`) will be escalated. Every transaction is aggregated into one feature row per alert. The final prediction is a rank blend of LightGBM (5 folds × 3 seeds) and a quantile-transformed logistic regression.

**Requirements:** pandas, numpy, pyarrow, lightgbm, scikit-learn, matplotlib. Place the data files next to this notebook.

In [1]:
TEAM_ID = "2FA74A79"
DATA = "."
import json, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")
SEEDS = (0, 1, 2)

## 1. Load data

In [2]:
tr_sig = pd.read_csv(f"{DATA}/train_signals.csv", parse_dates=["signal_sanasi"])
te_sig = pd.read_csv(f"{DATA}/test_signals.csv", parse_dates=["signal_sanasi"])
tr_tx = pd.read_parquet(f"{DATA}/train_transactions.parquet")
te_tx = pd.read_parquet(f"{DATA}/test_transactions.parquet")
print(tr_sig.shape, te_sig.shape, tr_tx.shape, te_tx.shape)
print("escalation rate:", tr_sig.eskalatsiya.mean().round(4))

(14000, 3) (6000, 2) (6987663, 5) (3027575, 5)
escalation rate: 0.1718


## 2. EDA summary (numbers exported for the EDA website)

In [3]:
t = tr_tx.merge(tr_sig, on="signal_id")
t["dd"] = (t.signal_sanasi - t.tranzaksiya_vaqti).dt.total_seconds() / 86400
eda = {}
eda["n_train"], eda["n_test"] = len(tr_sig), len(te_sig)
eda["n_tx_train"], eda["n_tx_test"] = len(tr_tx), len(te_tx)
eda["pos_rate"] = float(tr_sig.eskalatsiya.mean())
eda["tx_per_signal"] = t.groupby("signal_id").size().describe().round(1).to_dict()
eda["rate_by_quarter"] = tr_sig.groupby(tr_sig.signal_sanasi.dt.to_period("Q").astype(str)).eskalatsiya.agg(["mean", "size"]).round(4).reset_index().values.tolist()
# activity by days before alert (per-signal counts, by class)
bins = [-1, 0, 1, 3, 7, 14, 30, 60, 90, 180]
t["b"] = pd.cut(t.dd, bins)
cnt = t.groupby(["b", "eskalatsiya"], observed=False).size().unstack().div(tr_sig.eskalatsiya.value_counts(), axis=1)
eda["activity_windows"] = [[str(i), float(r[0]), float(r[1])] for i, r in cnt.iterrows()]
amt = t.groupby(["b", "eskalatsiya"], observed=False).miqdor_indeksi.mean().unstack()
eda["amount_windows"] = [[str(i), float(r[0]), float(r[1])] for i, r in amt.iterrows()]
# daily activity over the 180-day window
t["day"] = np.floor(t.dd).clip(-1, 180).astype(int)
daily = t.groupby(["day", "eskalatsiya"]).size().unstack().div(tr_sig.eskalatsiya.value_counts(), axis=1)
eda["daily"] = [[int(i), float(r[0]), float(r[1])] for i, r in daily.iterrows()]
eda["hourly_lastday"] = t[(t.dd > 0) & (t.dd <= 1)].tranzaksiya_vaqti.dt.hour.value_counts(normalize=True).sort_index().round(4).to_dict()
eda["hourly_hist"] = t[t.dd > 1].tranzaksiya_vaqti.dt.hour.value_counts(normalize=True).sort_index().round(4).to_dict()
# type / direction mix and amount per type
eda["type_mix"] = pd.crosstab(t.tranzaksiya_turi, t.eskalatsiya, normalize="columns").round(4).reset_index().values.tolist()
eda["dir_mix"] = pd.crosstab(t.kirim_chiqim, t.eskalatsiya, normalize="columns").round(4).reset_index().values.tolist()
eda["amt_by_type_dir"] = t.groupby(["tranzaksiya_turi", "kirim_chiqim"]).miqdor_indeksi.agg(["mean", "median", "size"]).round(3).reset_index().values.tolist()
# amount density and likelihood ratio (escalated / dismissed)
edges = np.arange(-3, 5.01, 0.25)
lr = {}
for tt in ["all", "bank_otkazmasi", "karta", "naqd", "xalqaro"]:
    x = t if tt == "all" else t[t.tranzaksiya_turi == tt]
    h0, _ = np.histogram(x[x.eskalatsiya == 0].miqdor_indeksi, edges, density=True)
    h1, _ = np.histogram(x[x.eskalatsiya == 1].miqdor_indeksi, edges, density=True)
    lr[tt] = [[float(e), float(a), float(b)] for e, a, b in zip(edges[:-1], h0, h1)]
eda["amount_density"] = lr
del t

## 3. Feature engineering
Most features come from `build_features`: windowed counts and amounts (1–180 days), type×direction profiles, gaps, daily/monthly dynamics and burst features. `extra` adds per-type amount quantiles and recent-vs-history ratios.

In [4]:
import numpy as np
import pandas as pd

WINDOWS = [1, 3, 7, 14, 30, 60, 90, 180]


def build_features(tx: pd.DataFrame, sig: pd.DataFrame) -> pd.DataFrame:
    """Aggregate a signal's transaction history into one row of features."""
    t = tx.merge(sig[["signal_id", "signal_sanasi"]], on="signal_id")
    t["dd"] = (t.signal_sanasi - t.tranzaksiya_vaqti).dt.total_seconds() / 86400
    t["amt"] = t.miqdor_indeksi
    t["out"] = (t.kirim_chiqim == "chiqim").astype(np.int8)
    t["hour"] = t.tranzaksiya_vaqti.dt.hour
    t["night"] = t.hour.isin([0, 1, 2, 3, 4, 5]).astype(np.int8)
    t["wkend"] = (t.tranzaksiya_vaqti.dt.dayofweek >= 5).astype(np.int8)
    t["big"] = (t.amt > 2).astype(np.int8)
    t["signed"] = np.where(t.out == 1, -1, 1) * np.exp(t.amt)
    t["eamt"] = np.exp(t.amt)
    t = t.sort_values(["signal_id", "tranzaksiya_vaqti"])
    g = t.groupby("signal_id")
    F = pd.DataFrame(index=sig.signal_id)

    # overall
    F["n_all"] = g.size()
    F["n_after"] = t[t.dd <= 0].groupby("signal_id").size()
    F["span_days"] = (g.tranzaksiya_vaqti.max() - g.tranzaksiya_vaqti.min()).dt.total_seconds() / 86400
    F["first_dd"] = g.dd.max()
    F["last_dd"] = g.dd.min()
    for c in ["amt"]:
        a = g[c].agg(["mean", "std", "min", "max", "median", "skew"])
        a.columns = [f"{c}_{x}" for x in a.columns]
        F = F.join(a)
    q = g.amt.quantile([0.1, 0.9]).unstack()
    F["amt_q10"], F["amt_q90"] = q[0.1], q[0.9]
    F["out_frac"] = g.out.mean()
    F["night_frac"] = g.night.mean()
    F["wkend_frac"] = g.wkend.mean()
    F["big_frac"] = g.big.mean()
    F["hour_mean"] = g.hour.mean()
    F["hour_std"] = g.hour.std()
    F["net_flow"] = g.signed.sum()
    F["eamt_sum"] = g.eamt.sum()

    # type x direction
    t["td"] = t.tranzaksiya_turi + "_" + t.kirim_chiqim
    ct = t.pivot_table(index="signal_id", columns="td", values="amt", aggfunc=["size", "mean", "max"])
    ct.columns = [f"td_{a}_{b}" for a, b in ct.columns]
    F = F.join(ct)
    for c in [c for c in F.columns if c.startswith("td_size_")]:
        F[c.replace("size", "frac")] = F[c].fillna(0) / F.n_all

    # time windows
    for w in WINDOWS:
        s = t[(t.dd > 0) & (t.dd <= w)]
        gs = s.groupby("signal_id")
        F[f"n_{w}"] = gs.size()
        F[f"amt_mean_{w}"] = gs.amt.mean()
        F[f"amt_max_{w}"] = gs.amt.max()
        F[f"amt_std_{w}"] = gs.amt.std()
        F[f"eamt_{w}"] = gs.eamt.sum()
        F[f"net_{w}"] = gs.signed.sum()
        F[f"out_frac_{w}"] = gs.out.mean()
        F[f"n_types_{w}"] = gs.tranzaksiya_turi.nunique()
        for tt in ["naqd", "xalqaro", "bank_otkazmasi", "karta"]:
            F[f"n_{tt}_{w}"] = s[s.tranzaksiya_turi == tt].groupby("signal_id").size()
        F[f"n_{w}"] = F[f"n_{w}"].fillna(0)
    # ratios recent vs baseline (per-day rate)
    base_rate = (F.n_180 - F.n_30) / 150
    base_amt = t[(t.dd > 30)].groupby("signal_id").amt.mean()
    for w in [1, 3, 7, 14, 30]:
        F[f"rate_ratio_{w}"] = (F[f"n_{w}"] / w) / (base_rate + 1e-3)
        F[f"amt_diff_{w}"] = F[f"amt_mean_{w}"] - base_amt
    F["burst_n"] = t[(t.dd > 0) & (t.dd <= 1 / 24)].groupby("signal_id").size()
    F["burst_amt"] = t[(t.dd > 0) & (t.dd <= 1 / 24)].groupby("signal_id").amt.mean()

    # gaps between consecutive transactions
    t["gap"] = t.groupby("signal_id").tranzaksiya_vaqti.diff().dt.total_seconds() / 3600
    gg = t.groupby("signal_id").gap
    F["gap_mean"] = gg.mean()
    F["gap_med"] = gg.median()
    F["gap_min"] = gg.min()
    F["gap_max"] = gg.max()
    F["gap_std"] = gg.std()
    F["gap_lt1m"] = (t.gap < 1 / 60).groupby(t.signal_id).mean()

    # daily activity profile
    t["day"] = np.floor(t.dd).astype(int)
    d = t[t.dd > 1].groupby(["signal_id", "day"]).agg(n=("amt", "size"), e=("eamt", "sum"))
    dg = d.groupby("signal_id")
    F["active_days"] = dg.size()
    F["daily_n_mean"] = dg.n.mean()
    F["daily_n_max"] = dg.n.max()
    F["daily_n_std"] = dg.n.std()
    F["daily_e_max"] = dg.e.max()
    F["daily_e_std"] = dg.e.std()

    # monthly trend of counts and amounts
    t["mon"] = np.clip((t.dd // 30).astype(int), 0, 5)
    m = t[t.dd > 0].pivot_table(index="signal_id", columns="mon", values="amt", aggfunc=["size", "mean"])
    x = np.arange(6)[::-1]
    cnt = m["size"].reindex(columns=range(6)).fillna(0).values
    mam = m["mean"].reindex(columns=range(6)).values
    xc = x - x.mean()
    F["cnt_slope"] = (cnt - cnt.mean(1, keepdims=True)) @ xc / (xc @ xc)
    F["amt_slope"] = np.nan_to_num(mam - np.nanmean(mam, 1, keepdims=True)) @ xc / (xc @ xc)

    # amount repeat / round patterns
    F["amt_nuniq_frac"] = g.amt.nunique() / F.n_all
    F["sig_month"] = sig.set_index("signal_id").signal_sanasi.dt.month
    F["sig_dow"] = sig.set_index("signal_id").signal_sanasi.dt.dayofweek
    return F


In [5]:
def extra(tx, sig):
    t=tx.merge(sig[['signal_id','signal_sanasi']],on='signal_id')
    t['dd']=(t.signal_sanasi-t.tranzaksiya_vaqti).dt.total_seconds()/86400
    t['td']=t.tranzaksiya_turi+'_'+t.kirim_chiqim
    out=[]
    for key in ['td','tranzaksiya_turi','kirim_chiqim']:
        q=t.groupby(['signal_id',key]).miqdor_indeksi.quantile([.05,.25,.5,.75,.95]).unstack([1,2])
        q.columns=[f'q_{a}_{b}' for a,b in q.columns]; out.append(q)
        sd=t.groupby(['signal_id',key]).miqdor_indeksi.agg(['std','skew']).unstack(); sd.columns=[f'{a}_{b}' for a,b in sd.columns]; out.append(sd)
    # recent 30d vs older per type mean
    r=t[t.dd<=30].groupby(['signal_id','tranzaksiya_turi']).miqdor_indeksi.mean().unstack()
    o=t[t.dd>30].groupby(['signal_id','tranzaksiya_turi']).miqdor_indeksi.mean().unstack()
    out.append((r-o).add_prefix('recent_minus_old_'))
    rn=t[t.dd<=30].groupby(['signal_id','tranzaksiya_turi']).size().unstack()
    on=t[t.dd>30].groupby(['signal_id','tranzaksiya_turi']).size().unstack()
    out.append((rn/30/(on/150)).add_prefix('rate_ratio_type_'))
    return pd.concat(out,axis=1).reindex(sig.signal_id)


In [6]:
X = build_features(tr_tx, tr_sig).join(extra(tr_tx, tr_sig)).replace([np.inf, -np.inf], np.nan)
Xt = build_features(te_tx, te_sig).join(extra(te_tx, te_sig)).replace([np.inf, -np.inf], np.nan)[X.columns]
y = tr_sig.eskalatsiya.values
print(X.shape, Xt.shape)

(14000, 284) (6000, 284)


## 4. Models: 5-fold CV, 3 seeds

In [7]:
P = dict(n_estimators=600, learning_rate=0.01, num_leaves=7, min_child_samples=100, subsample=0.7,
         subsample_freq=1, colsample_bytree=0.3, reg_lambda=10, verbose=-1)
rk = lambda a: pd.Series(a).rank(pct=True).values
oof_g, oof_l = np.zeros(len(y)), np.zeros(len(y))
pred_g, pred_l = np.zeros(len(Xt)), np.zeros(len(Xt))
imp = pd.Series(0.0, X.columns)
for sd in SEEDS:
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=sd).split(X, y):
        g = lgb.LGBMClassifier(**P, random_state=sd).fit(X.iloc[tr], y[tr])
        oof_g[va] += rk(g.predict_proba(X.iloc[va])[:, 1]); pred_g += rk(g.predict_proba(Xt)[:, 1])
        imp += g.booster_.feature_importance("gain")
        l = make_pipeline(SimpleImputer(strategy="median"), QuantileTransformer(output_distribution="normal"),
                          LogisticRegression(C=0.005, max_iter=2000)).fit(X.iloc[tr], y[tr])
        oof_l[va] += rk(l.predict_proba(X.iloc[va])[:, 1]); pred_l += rk(l.predict_proba(Xt)[:, 1])
W = 0.4
oof = (1 - W) * rk(oof_g) + W * rk(oof_l)
print("OOF AUC  LGBM %.4f | LogReg %.4f | blend %.4f" % (roc_auc_score(y, oof_g), roc_auc_score(y, oof_l), roc_auc_score(y, oof)))
eda["cv"] = {"lgbm": roc_auc_score(y, oof_g), "logreg": roc_auc_score(y, oof_l), "blend": roc_auc_score(y, oof)}
eda["importance"] = (imp / imp.sum()).nlargest(20).round(4).reset_index().values.tolist()
uni = pd.Series({c: roc_auc_score(y, X[c].fillna(X[c].median())) for c in X.columns})
eda["univariate_auc"] = uni.sub(0.5).abs().nlargest(15).add(0.5).round(4).reset_index().values.tolist()
eda["univariate_dir"] = {c: float(uni[c]) for c, _ in eda["univariate_auc"]}

OOF AUC  LGBM 0.6471 | LogReg 0.6399 | blend 0.6490


## 5. Submission

In [8]:
pred = (1 - W) * rk(pred_g) + W * rk(pred_l)
pred = (pred - pred.min()) / (pred.max() - pred.min())   # rank score in [0, 1]; AUC depends on ranks only
sub = pd.DataFrame({"signal_id": te_sig.signal_id, "ehtimollik": pred.round(6)})
assert sub.signal_id.is_unique and len(sub) == len(te_sig) and sub.ehtimollik.between(0, 1).all()
sub.to_csv(f"team_{TEAM_ID}.csv", index=False)
json.dump(eda, open("eda_stats.json", "w"), default=float)
sub.head()

,signal_id,ehtimollik
0,SG_000001,0.245562
1,SG_000007,0.656123
2,SG_000009,0.159626
3,SG_000010,0.370037
4,SG_000011,0.603770
